# H-001 Backtest: 12-1 vs 12-7 momentum บน SET100

Notebook นี้รัน `scripts/backtest_momentum.py` แบบเดียวกับใน terminal (EP4) โดยไม่ต้องติดตั้งอะไรในเครื่อง

- ทุกการรันจริงคือ **1 trial** สคริปต์จะเขียน `experiments/E-xxx.md` ใหม่และเพิ่ม `trial_no` ทีละ 1
  (ใน repo มี trial ของผู้สอนอยู่แล้ว trial ของคุณจะต่อจากนั้น)
- ตัวเลขทั้งหมดมาจากสคริปต์ ไม่ได้มาจาก LLM
- รายชื่อ SET100 เป็นรายชื่อ**ปัจจุบัน** จึงมี survivorship bias โดยตั้งใจ (Reviewer จะจับใน EP5)

> เพื่อการศึกษา ไม่ใช่คำแนะนำการลงทุน

In [ ]:
# 1) clone repo (ข้ามถ้าเปิด notebook จากใน repo อยู่แล้ว)
import os, sys
REPO_URL = "https://github.com/Migchw/alpha-research-wiki.git"
if not os.path.exists("scripts/backtest_momentum.py"):
    if os.path.exists("../scripts/backtest_momentum.py"):
        os.chdir("..")
    else:
        !git clone -q {REPO_URL}
        os.chdir("alpha-research-wiki")
print(os.getcwd())

In [ ]:
# 2) ติดตั้ง dependency
!{sys.executable} -m pip install -q -r requirements.txt

In [ ]:
# 3) ตรวจ signal window และสูตรค่าธรรมเนียมด้วยข้อมูลสังเคราะห์ (ไม่ใช้ network)
!{sys.executable} scripts/test_backtest_momentum.py

In [ ]:
# 4) ดูผลก่อนโดยไม่นับเป็น trial (--dry-run ไม่เขียนไฟล์)
!{sys.executable} scripts/backtest_momentum.py --hypothesis H-001 --lookback 12-7 --dry-run | tail -n 3

In [ ]:
# 5) รันจริง in-sample ทั้งสอง lookback → เขียน experiments/E-xxx.md
!{sys.executable} scripts/backtest_momentum.py --hypothesis H-001 --lookback 12-1
!{sys.executable} scripts/backtest_momentum.py --hypothesis H-001 --lookback 12-7

In [ ]:
# 6) out-of-sample ถูกล็อกไว้จนกว่า Reviewer จะให้ in-sample ผ่าน (คาดว่าจะเห็น REFUSED)
!{sys.executable} scripts/backtest_momentum.py --hypothesis H-001 --lookback 12-7 --sample out_of_sample

In [ ]:
# 7) lint vault หลังรัน (ต้อง PASS)
!{sys.executable} scripts/wiki_lint.py

In [ ]:
# 8) Experiment ledger แบบย่อ อ่านจาก frontmatter
import glob, yaml, pandas as pd
rows = []
for path in sorted(glob.glob("experiments/E-*.md")):
    text = open(path, encoding="utf-8").read()
    rows.append(yaml.safe_load(text.split("---", 2)[1]))
cols = ["id", "trial_no", "lookback", "sample", "sharpe", "t_stat", "ann_return", "max_dd", "turnover", "reviewer_verdict"]
pd.DataFrame(rows)[cols]

## คำถามชวนคิดก่อนไป EP5
1. Q5 − Q1 ของ 12-7 ต่างจาก 12-1 อย่างไร ตรงกับ [[novy-marx-2012]] หรือไม่
2. ตอนนี้ H-001 ใช้ไปกี่ trial แล้ว Sharpe ที่เห็นควรถูกลดทอนเท่าไร (deflated Sharpe, Module 8)
3. รายชื่อ SET100 ปัจจุบันทำให้ผลดูดีเกินจริงในทิศไหน